# AI-Lake + Trino

Trino reads AI-Lake tables via the standard **Iceberg connector** —
no AI-Lake plugin required for tabular queries.

**Prerequisites:** start the engines compose overlay:
```bash
docker compose \
  -f tests/docker/compose-demo.yml \
  --profile engines \
  up -d
```

**What this notebook covers:**
1. Connect to Trino via the Python `trino` driver
2. Discover catalogs, schemas, tables
3. SQL queries: scan, filter, aggregation
4. `embedding` column as `varbinary`
5. Iceberg metadata queries (`$snapshots`, `$manifests`)

In [ ]:
import os
import trino

TRINO_HOST = os.environ.get('TRINO_HOST', 'trino')
TRINO_PORT = int(os.environ.get('TRINO_PORT', '8080'))

print(f'Connecting to Trino at {TRINO_HOST}:{TRINO_PORT} ...')

conn = trino.dbapi.connect(
    host=TRINO_HOST,
    port=TRINO_PORT,
    user='ailake',
    catalog='ailake',
    schema='default',
)
cur = conn.cursor()

def q(sql):
    """Execute SQL and return rows."""
    cur.execute(sql)
    return cur.fetchall()

def qdf(sql):
    """Execute SQL and return a pandas DataFrame."""
    import pandas as pd
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description]
    return pd.DataFrame(rows, columns=cols)

print('Connected.')

In [ ]:
# ── Pre-flight check — verify Trino is reachable ─────────────────────────────
import os, socket

TRINO_HOST = os.environ.get('TRINO_HOST', 'trino')
TRINO_PORT = int(os.environ.get('TRINO_PORT', '8080'))

try:
    with socket.create_connection((TRINO_HOST, TRINO_PORT), timeout=5):
        print(f"OK: Trino reachable at {TRINO_HOST}:{TRINO_PORT}")
except OSError:
    raise RuntimeError(
        f"\n\nTrino is not reachable at {TRINO_HOST}:{TRINO_PORT}.\n"
        "Start the engines overlay first:\n\n"
        "  docker compose \\\n"
        "    -f tests/docker/compose-demo.yml \\\n"
        "    --profile engines \\\n"
        "    up -d\n"
    )

## 1. Discover catalogs and tables

In [ ]:
print('Catalogs:', q('SHOW CATALOGS'))
print('Schemas :', q('SHOW SCHEMAS FROM ailake'))
print('Tables  :', q('SHOW TABLES FROM ailake.default'))

## 2. Schema inspection

In [ ]:
# embedding is varbinary — standard Parquet FIXED_LEN_BYTE_ARRAY read as bytes
qdf('DESCRIBE ailake.default."table"')

## 3. Basic scan

In [ ]:
count = q('SELECT count(*) FROM ailake.default."table"')[0][0]
print(f'Total rows: {count}')

In [ ]:
qdf("""
    SELECT text,
           length(embedding) AS embedding_bytes
    FROM ailake.default."table"
    LIMIT 5
""")

## 4. Filtered query + aggregation

In [ ]:
df = qdf("""
    SELECT text
    FROM ailake.default."table"
    WHERE text LIKE '%vector search%'
""")
print(f"Rows about 'vector search': {len(df)}")
df.head(3)

In [ ]:
qdf("""
    SELECT
        count(*)               AS total_rows,
        avg(length(text))      AS avg_text_chars,
        avg(length(embedding)) AS avg_embedding_bytes
    FROM ailake.default."table"
""")

## 5. Iceberg metadata — snapshots and manifests

In [ ]:
# Trino Iceberg connector exposes system tables via the $ suffix
qdf('SELECT snapshot_id, committed_at, operation FROM "ailake"."default"."table$snapshots"')

In [ ]:
qdf("""
    SELECT
        file_path,
        record_count,
        file_size_in_bytes
    FROM "ailake"."default"."table$files"
""")

## 6. Table properties — AI-Lake custom metadata via Trino

AI-Lake stores vector configuration in Iceberg table properties.
Trino exposes these via the `$properties` system table.

> `ailake.embedding-model` appears here when the table was written with
> `embedding_model=` — e.g. `ailake.open_table(..., embedding_model="text-embedding-3-small")`.


In [ ]:
props = qdf('SELECT * FROM "ailake"."default"."table$properties"')
# Filter AI-Lake properties
ailake_props = props[props['key'].str.startswith('ailake.')]
print(f'AI-Lake properties in Trino Iceberg catalog:')
ailake_props


## 7. File-level manifest statistics

Trino exposes per-file record counts and sizes from the Iceberg manifest via `$files`.


In [ ]:
sql = (
    "SELECT split_part(file_path, '/', -1) AS filename,"
    " record_count, file_size_in_bytes,"
    " round(cast(file_size_in_bytes AS double) / record_count, 1) AS bytes_per_row"
    ' FROM \"ailake\".\"default\".\"table$files\"'
)
qdf(sql)


## 8. Manifest files

`$manifests` lists the Avro manifest files that compose each Iceberg snapshot.
Trino reads these to discover which Parquet files to scan.


In [ ]:
qdf('SELECT path, length, partition_spec_id, added_files_count FROM "ailake"."default"."table$manifests"')


## 9. Embedding model tracking via Trino `$properties`

When a table has `ailake.embedding-model` set, it appears in the Trino
`$properties` system table. Use it to audit which model version was used
and to validate before running `migrate_embeddings()` from the SDK.

In [ ]:
# Filter embedding-model properties specifically
emb_df = qdf(
    'SELECT "key", "value" FROM "ailake"."default"."table$properties"'
    ' WHERE "key" LIKE \'ailake.embedding%\''
)
if len(emb_df) > 0:
    print('Embedding model properties visible in Trino:')
    print(emb_df.to_string(index=False))
else:
    print('(ailake.embedding-model not set — write with embedding_model= to populate)')


## Key takeaway

Trino queries AI-Lake tables through the **standard Iceberg connector** —
the Parquet files and Iceberg metadata are fully compatible with Trino 446.
The `embedding` column appears as `varbinary`.

For vector similarity search, use the AI-Lake SDK (`ailake.search()`) or deploy
the `trino-plugin` JAR which adds `vector_search()` as a Trino function.